# Breast Cancer Classification with scikit-learn

This notebook demonstrates classification of the Breast Cancer Wisconsin (Diagnostic) dataset
using several machine learning algorithms from scikit-learn.

**Algorithms evaluated:**
- Logistic Regression
- Support Vector Machine (linear kernel)
- Support Vector Machine (RBF kernel)
- Decision Tree
- Random Forest
- K-Nearest Neighbors (KNN)

## 1. Imports

Import all required libraries.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

## 2. Load the Dataset

Load the [Breast Cancer Wisconsin (Diagnostic) dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset)
directly from `sklearn.datasets`. The dataset contains 569 samples with 30 numeric features
describing characteristics of cell nuclei, and a binary target: **malignant (0)** or **benign (1)**.

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Target classes: {data.target_names}")
print(f"Class distribution: {dict(zip(data.target_names, [(y == i).sum() for i in range(len(data.target_names))]))}") 

## 3. Train/Test Split

Split the data into **80% training** and **20% testing** sets.
- `stratify=y` ensures the class proportions are preserved in both splits.
- `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Test set size     : {X_test.shape[0]} samples")

## 4. Feature Scaling

Standardize the features using `StandardScaler` (zero mean, unit variance).
- **Fit** only on the training set to avoid data leakage.
- **Transform** both the training and test sets using the fitted scaler.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Features standardized (fitted on training set only).")

## 5. Define Classifiers

Define all six classifiers with fixed `random_state` where applicable for reproducibility.

In [ ]:
classifiers = {
    "Logistic Regression" : LogisticRegression(max_iter=10000, random_state=42),
    "SVM (Linear Kernel)" : SVC(kernel="linear", random_state=42),
    "SVM (RBF Kernel)"    : SVC(kernel="rbf",    random_state=42),
    "Decision Tree"       : DecisionTreeClassifier(random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN"                 : KNeighborsClassifier(),
}

## 6. Train and Evaluate

Train each classifier on the scaled training data and compute:
- **Training accuracy** – accuracy on the training set (measures how well the model fits the training data).
- **Testing accuracy** – accuracy on the held-out test set (measures generalisation).

In [ ]:
results = []

for name, clf in classifiers.items():
    # Train
    clf.fit(X_train_scaled, y_train)

    # Predict
    train_pred = clf.predict(X_train_scaled)
    test_pred  = clf.predict(X_test_scaled)

    # Accuracies
    train_acc = accuracy_score(y_train, train_pred)
    test_acc  = accuracy_score(y_test,  test_pred)

    results.append({
        "Model"            : name,
        "Train Accuracy"   : round(train_acc, 4),
        "Test Accuracy"    : round(test_acc,  4),
    })

    print(f"{name}")
    print(f"  Train Accuracy : {train_acc:.4f}")
    print(f"  Test  Accuracy : {test_acc:.4f}")
    print()

## 7. Results Summary Table

Collect all results into a pandas DataFrame for a clear, side-by-side comparison.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

print("=" * 55)
print(f"{'Model':<25} {'Train Acc':>10} {'Test Acc':>10}")
print("=" * 55)
for _, row in results_df.iterrows():
    print(f"{row['Model']:<25} {row['Train Accuracy']:>10.4f} {row['Test Accuracy']:>10.4f}")
print("=" * 55)

# Display the DataFrame
results_df